In [ ]:
import sys
import torch
import functools
import matplotlib.pyplot as plt
import argparse, yaml, os
import torch.optim.lr_scheduler as lr_scheduler
import torchvision.transforms as transforms
import seaborn as sns
import pandas as pd
import tqdm
import glob

from torch.optim import Adam
from torch.utils.data import DataLoader
from torchvision.datasets import MNIST
from types import SimpleNamespace

from IPython.display import clear_output

sys.path.append('/om2/user/jmhicks/projects/TextureStreaming/code/')

from chexture_choolbox.auditorytexture.statistics_sets import (
    STAT_SET_FULL_MCDERMOTTSIMONCELLI as statistics_dict
)
from chexture_choolbox.auditorytexture.texture_model import TextureModel
from chexture_choolbox.auditorytexture.helpers import FlattenStats
from texture_prior.params import model_params as model_params_tm
from texture_prior.params import statistics_set, texture_dataset
from texture_prior.utils import path

sys.path.append('../utls/')
sys.path.append('../src/model/')
sys.path.append("/om2/user/bjmedina/auditory-memory/memory/")
import DistanceMemoryModel
import encoders

from utls.loading import load_results, load_results_with_isi0_exclusion, load_results_with_isi0_dprime_exclusion, move_sequences_to_used, load_results_with_exclusion
from utls.dprime import recompute_dprime_by_isi_per_subject
from utls.reliability import compute_itemwise_split_half_reliability
from utls.plotting import plot_dprime_by_isi, plot_itemwise_split_half_scatter_df, ensure_dir


sounds_list = glob.glob("/mindhive/mcdermott/www/mturk_stimuli/bjmedina/mem_exp_atexts_2025/*wav")
device = 'cuda'
# get soem textures
pc_dims = 256

pc_texture_model = encoders.AudioTextureEncoderPCA(
    statistics_dict=statistics_set.statistics,
    pc_dims=pc_dims,
    model_params=model_params_tm,
    sr=20000,
    rms_level=0.05,
    duration=2.0,
    device=device
)

def compute_likelihood(score_model, input_stats, ckpt):
    """Computing the actual "prior" value
    If the score function can be treated as the vector field govering the temporal evolution
    of $x_t$ then we can integrate the ODE and apply a change of basis to evaluate the prior
    """
    score_model.load_state_dict(torch.load(ckpt))
    input_stats = input_stats.to(device)

    _, bpd = ode_likelihood(input_stats, score_model, marginal_prob_std_fn,
                            diffusion_coeff_fn,
                            input_stats.shape[0], device=device, eps=1e-5)
    
    return bpd

def parse(d):
  x = SimpleNamespace()
  _ = [setattr(x, k, parse(v)) if isinstance(v, dict) else setattr(x, k, v) for k, v in d.items() ]    
  return x

In [ ]:
pc_texture_model

In [ ]:
# parser.add_argument('--config', type=str, required=True, help='model + data configuration')
# parser.add_argument('--train', action='store_true', help='train the score-based model')
# parser.add_argument('--sample', action='store_true', help='sample from a trained model')
# parser.add_argument('--likelihood_eval', action='store_true', help='evaluate the data likelihood')
# parser.add_argument('--restart', action='store_true')
# parser.add_argument('--mode', type=str, default='textures', choices=['textures', 'mixtures'])

# sys.path.append('/om2/user/lakshmin/audio-prior/')
# from models import ScoreNetAudio, ScoreNetTexture1D, ScoreNetAudioV2
# from utils import synthesis, projection, audio
# from utils.sde_utils import *

import sys
import importlib.util
import os

# Add the new path
audio_prior_path = '/om2/user/lakshmin/audio-prior/'
sys.path.insert(0, audio_prior_path)  # insert at front of sys.path

from models import ScoreNetAudio, ScoreNetTexture1D, ScoreNetAudioV2
from utils import synthesis, projection, audio
from utils.sde_utils import *

config = "/om2/user/bjmedina/auditory-memory/memory/assets/bryan.yaml"
train = False
sample = False
likelihood_eval = True
restart = False
mode = 'textures'

device = 'cuda'

In [ ]:
df = yaml.safe_load(open(config))
cfg = parse(df)

print(cfg.data)

In [ ]:
score_model = torch.nn.DataParallel(
                    ScoreNetAudioV2(
                        marginal_prob_std=marginal_prob_std_fn, 
                        channels=cfg.model.channels, 
                        embed_dim=cfg.model.embed_dim
                        )
                    )

score_model = score_model.to(device)

ckpt_path = cfg.model.ckpt_path.format(cfg.data.n_pcs, mode)
ckpt_path = "/om2/user/lakshmin/audio-prior/" + ckpt_path
if 'SLURM_RESTART_COUNT' in os.environ.keys() or restart:
    score_model.load_state_dict(torch.load(ckpt_path))

print(ckpt_path)

In [ ]:
exp_sounds_list = glob.glob("/mindhive/mcdermott/www/mturk_stimuli/bjmedina/mem_exp_atexts_2025/*wav")
ALL_SOUNDS = glob.glob("/om2/data/public/audioset/wavs/unbalanced_train_segments_downloads/unbalanced_train_segments_downloads_*/*wav")
print(len(ALL_SOUNDS))

# results = set(glob.glob("/mindhive/mcdermott/www/bjmedina/experiments/bolivia_2025/results/isi_16/ind-nature-len120/*csv"))
# results = list(results)

tasks = ["ind-nature-len120" ,"global-music-len120", "atexts-len120", "nhs-region-len120"]
which_task = tasks[2] # "global-music-len120", "atexts-len120" "nhs-region-len120"
base_path = "/mindhive/mcdermott/www/mturk_stimuli/bjmedina/{}/sequences/isi_16/len120/"
seqs_paths = {"ind-nature-len120": "mem_exp_ind-nature_2025", 
              "global-music-len120": "global-music-2025-n_80",
              "atexts-len120": "mem_exp_atexts_2025",
              "nhs-region-len120": "nhs-region-n_80"}


hr_task_name = {"ind-nature-len120": "Industrial and Nature", 
              "global-music-len120": "Globalized Music",
              "atexts-len120": "Auditory Textures",
              "nhs-region-len120": " 'Natural History of Song' "}

exps, seqs, fnames = load_results_with_exclusion(f"/mindhive/mcdermott/www/bjmedina/experiments/bolivia_2025/results/isi_16/{which_task}",
                                                    min_dprime=2,
                                                    min_trials=120,
                                                    skip_len60=True,
                                                    verbose=False,
                                                    return_skipped=False)



move_sequences_to_used(base_path.format(seqs_paths[which_task]), seqs)

print("Number of participants used in analysis:", len(exps))


safe_name = which_task.lower().replace(" ", "_")  # e.g., "globalized_music"
save_dir = os.path.join("/om2/user/bjmedina/auditory-memory/memory/figures/human-results/isi-16-only", safe_name)

ensure_dir(save_dir)
print(save_dir)

In [ ]:
x = recompute_dprime_by_isi_per_subject(exps)
# Assuming `x` is your DataFrame
valid_isi_values = [-1, 0, 16]
x_filtered = x[x['isi'].isin(valid_isi_values)].copy()
x = x_filtered

results_long = compute_itemwise_split_half_reliability(exps, min_isi=16, max_isi=16)
hits = results_long['itemwise_responses']['hits']
false_alarms  = results_long['itemwise_responses']['false_alarms']

stim_fas = [false_alarms[stim].mean(axis=0) for stim in false_alarms.columns.tolist()]
stim_hits = [hits[stim].mean(axis=0) for stim in false_alarms.columns.tolist()]
stim_nlls = [compute_likelihood(score_model, input_stats=pc_texture_model("/mindhive/mcdermott/www/mturk_stimuli/bjmedina/mem_exp_atexts_2025/"+stim).view(1, 1, 1, 256), ckpt=ckpt_path)[0] for stim in false_alarms.columns.tolist()]
stim_nlls = [nll.detach().cpu().numpy() for nll in stim_nlls]


In [ ]:
from sklearn.neighbors import NearestNeighbors

def knn_log_density(X_probe, X_ref=None, k=50, exclude_self=True):
    """
    Estimate log-density of probe points using kNN distances.

    Args:
        X_probe : np.ndarray, shape (N, d)
            Probe feature matrix (items to score).
        X_ref   : np.ndarray, shape (M, d), optional
            Reference feature matrix (population for neighbors).
            If None, uses X_probe as reference.
        k       : int
            Number of neighbors to use.
        exclude_self : bool
            If True and X_probe is X_ref, exclude self-match.

    Returns:
        log_density : np.ndarray, shape (N,)
    """
    if X_ref is None:
        X_ref = X_probe
        same_ref = True
    else:
        same_ref = np.may_share_memory(X_probe, X_ref) or np.array_equal(X_probe, X_ref)

    # fit on reference set
    nbrs = NearestNeighbors(n_neighbors=k + (1 if (same_ref and exclude_self) else 0),
                            algorithm='auto').fit(X_ref)
    distances, indices = nbrs.kneighbors(X_probe)

    if same_ref and exclude_self:
        # drop the first neighbor (self-distance ~ 0)
        rk = distances[:, 1 + (k-1)] if k > 1 else distances[:, 1]
    else:
        rk = distances[:, -1]  # distance to k-th neighbor

    d = X_probe.shape[1]
    log_density = -d * np.log(rk + 1e-12)
    return log_density

rep = [ pc_texture_model("/mindhive/mcdermott/www/mturk_stimuli/bjmedina/mem_exp_atexts_2025/"+stim) for stim in false_alarms.columns.tolist() ]
x = torch.stack(rep).float()  # shape: [N, ...]
x = x.view(80, 256)
x_exp = x.cpu()

num_sounds = 1000
rep = [pc_texture_model(stim) for stim in ALL_SOUNDS[:num_sounds]]
x_other = torch.stack(rep).float()  # shape: [N, ...]
x_other = x_other.view(len(rep), 256)
x_other = x_other.cpu()

log_densities_exp = knn_log_density(x_exp, x_exp, k=50)
log_densities_other = knn_log_density(x_exp, x_other, k=50)

In [ ]:
import numpy as np
from scipy import stats

r_hit = stats.pearsonr(log_densities_exp, stim_hits)[0]
r_fa = stats.pearsonr(log_densities_exp, stim_fas)[0]


plt.scatter(log_densities_exp, stim_hits, label=f"Hit Rate: r_$hit$= {r_hit:.2f}", color='g', alpha=0.5)
plt.scatter(log_densities_exp, stim_fas, label=f"False Alarm Rate: r_$fa$= {r_fa:.2f}", color='r', alpha=0.5)

plt.legend()
plt.ylim([0,1.1])
plt.xlabel("log(P(x)): KNN")
plt.ylabel("Rate")
plt.title("Using Experimental sounds")

In [ ]:
r_hit = stats.pearsonr(log_densities_other, stim_hits)[0]
r_fa = stats.pearsonr(log_densities_other, stim_fas)[0]


plt.scatter(log_densities_other, stim_hits, label=f"Hit Rate: r_$hit$= {r_hit:.2f}", color='g', alpha=0.5)
plt.scatter(log_densities_other, stim_fas, label=f"False Alarm Rate: r_$fa$= {r_fa:.2f}", color='r', alpha=0.5)

plt.legend()
plt.ylim([0,1.1])
plt.xlabel("log(P(x)): KNN")
plt.ylabel("Rate")
plt.title("Using AudioSet sounds")

In [ ]:

r_hit = stats.pearsonr(stim_nlls, stim_hits)[0]
r_fa = stats.pearsonr(stim_nlls, stim_fas)[0]


plt.scatter(stim_nlls, stim_hits, label=f"Hit Rate: r_$hit$= {r_hit:.2f}", color='g', alpha=0.5)
plt.scatter(stim_nlls, stim_fas, label=f"False Alarm Rate: r_$fa$= {r_fa:.2f}", color='r', alpha=0.5)

plt.legend()
plt.ylim([0,1.1])
plt.xlabel("-log(P(x)): diffusion")
plt.ylabel("Rate")

In [ ]:
# from dataloader import TextureStatsDataset
# texture_dataset = TextureStatsDataset(
#                 config=cfg,                        
#                 device=device
#             )
# x = texture_dataset.get_random_batch(1024)


sounds_to_test = 2000
effective_num_sounds = sounds_to_test

rep = []

for sound in ALL_SOUNDS[:sounds_to_test]:
    vec = pc_texture_model(sound)

    try:
        if not torch.is_tensor(vec):
            vec = torch.tensor(vec)  # convert if it's a numpy array
        rep.append(vec)

        t = torch.tensor([0.01], device=vec.device).float()

        score = score_model(vec.view(1, 1, 1, 256), t)

        # 2. Flatten for norm computation per sample
        score_flat = score.view(score.size(0), -1)  # [B, 256]
        
        # 3. Normalize: divide by norm per sample
        norms = score_flat.norm(p=2, dim=1, keepdim=True) + 1e-8  # prevent division by zero
        unit_score_flat = score_flat / norms  # [B, 256]
        
        # 4. Reshape back to original format
        unit_score = unit_score_flat.view_as(score)  # [B, 1, 1, 256]
        #print(unit_score)

        #input()
        #clear_output(wait=True)

    
    except Exception as e:
        print(e)
        effective_num_sounds -= 1


# Now stack all into one tensor
x = torch.stack(rep).float()  # shape: [N, ...]
x = x.view(effective_num_sounds, 1, 1, 256)

bpd_textures = compute_likelihood(score_model, input_stats=x, ckpt=ckpt_path)

In [ ]:
plt.figure(figsize=(8, 5))

sns.histplot(bpd_textures.detach().cpu().numpy(), bins=50, color='tab:brown', label='Textures', stat='density', alpha=0.5, kde=True, element='step')

plt.title("NLL of Z-scored stats")
plt.xlabel(r'$-$log$p(x)$')
plt.ylabel('Density')
plt.legend()
plt.tight_layout()

In [ ]:
# from dataloader import TextureStatsDataset
# texture_dataset = TextureStatsDataset(
#                 config=cfg,                        
#                 device=device
#             )
# x = texture_dataset.get_random_batch(1024)


sounds_to_test = 2000
effective_num_sounds = sounds_to_test

rep = []

for sound in ALL_SOUNDS[:sounds_to_test]:
    vec = pc_texture_model(sound)

    try:
        if not torch.is_tensor(vec):
            vec = torch.tensor(vec)  # convert if it's a numpy array
        rep.append(vec)

        t = torch.tensor([0.01], device=vec.device).float()

        score = score_model(vec.view(1, 1, 1, 256), t)

        # 2. Flatten for norm computation per sample
        score_flat = score.view(score.size(0), -1)  # [B, 256]
        
        # 3. Normalize: divide by norm per sample
        norms = score_flat.norm(p=2, dim=1, keepdim=True) + 1e-8  # prevent division by zero
        unit_score_flat = score_flat / norms  # [B, 256]
        
        # 4. Reshape back to original format
        unit_score = unit_score_flat.view_as(score)  # [B, 1, 1, 256]
        #print(unit_score)

        #input()
        #clear_output(wait=True)

    
    except Exception as e:
        print(e)
        effective_num_sounds -= 1


# Now stack all into one tensor
x = torch.stack(rep).float()  # shape: [N, ...]
x = x.view(effective_num_sounds, 1, 1, 256)

bpd_textures = compute_likelihood(score_model, input_stats=x, ckpt=ckpt_path)

In [ ]:
plt.figure(figsize=(8, 5))

sns.histplot(bpd_textures.detach().cpu().numpy(), bins=50, color='tab:brown', label='Textures', stat='density', alpha=0.5, kde=True, element='step')

plt.title("NLL of PC stats")
plt.xlabel(r'$-$log$p(x)$')
plt.ylabel('Density')
plt.legend()
plt.tight_layout()

In [ ]:
# from dataloader import TextureStatsDataset
# texture_dataset = TextureStatsDataset(
#                 config=cfg,                        
#                 device=device
#             )
# x = texture_dataset.get_random_batch(1024)


sounds_to_test = 2000
effective_num_sounds = sounds_to_test

rep = []

t_something = 2000

exp_sounds = glob.glob("/mindhive/mcdermott/www/mturk_stimuli/bjmedina/mem_exp_atexts_2025/*wav")

for sound in exp_sounds:
    vec = pc_texture_model(sound)

    try:
        if not torch.is_tensor(vec):
            vec = torch.tensor(vec)  # convert if it's a numpy array
        rep.append(vec)

        t = torch.tensor([t_something], device=vec.device).float()

        

        score = score_model(vec.view(1, 1, 1, 256), t)

        # 2. Flatten for norm computation per sample
        score_flat = score.view(score.size(0), -1)  # [B, 256]
        
        # 3. Normalize: divide by norm per sample
        norms = score_flat.norm(p=2, dim=1, keepdim=True) + 1e-8  # prevent division by zero
        unit_score_flat = score_flat / norms  # [B, 256]
        
        # 4. Reshape back to original format
        unit_score = unit_score_flat.view_as(score)  # [B, 1, 1, 256]
        #print(unit_score)

        #input()
        #clear_output(wait=True)

    
    except Exception as e:
        print(e)
        effective_num_sounds -= 1


# Now stack all into one tensor
x = torch.stack(rep).float()  # shape: [N, ...]
x = x.view(len(exp_sounds), 1, 1, 256)

bpd_textures = compute_likelihood(score_model, input_stats=x, ckpt=ckpt_path)

In [ ]:
plt.figure(figsize=(8, 5))

sns.histplot(bpd_textures.detach().cpu().numpy(), bins=50, color='tab:brown', label='Textures', stat='density', alpha=0.5, kde=True, element='step')

plt.title(f"NLL of PC stats (exp. textures) t={t_something}")
plt.xlabel(r'$-$log$p(x)$')
plt.ylabel('Density')
plt.legend()
plt.tight_layout()

In [ ]:
plt.figure(figsize=(8, 5))

sns.histplot(bpd_textures.detach().cpu().numpy(), bins=50, color='tab:brown', label='Textures', stat='density', alpha=0.5, kde=True, element='step')

plt.title(f"NLL of PC stats (exp. textures) t={t_something}")
plt.xlabel(r'$-$log$p(x)$')
plt.ylabel('Density')
plt.legend()
plt.tight_layout()

In [ ]:
plt.figure(figsize=(8, 5))

sns.histplot(bpd_textures.detach().cpu().numpy(), bins=50, color='tab:brown', label='Textures', stat='density', alpha=0.5, kde=True, element='step')

plt.title(f"NLL of PC stats (exp. textures) t={t_something}")
plt.xlabel(r'$-$log$p(x)$')
plt.ylabel('Density')
plt.legend()
plt.tight_layout()

In [ ]:
plt.figure(figsize=(8, 5))

sns.histplot(bpd_textures.detach().cpu().numpy(), bins=50, color='tab:brown', label='Textures', stat='density', alpha=0.5, kde=True, element='step')

plt.title(f"NLL of PC stats (exp. textures) t={t_something}")
plt.xlabel(r'$-$log$p(x)$')
plt.ylabel('Density')
plt.legend()
plt.tight_layout()